[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nekrut/bda/blob/colab/lectures/lecture4a.ipynb)

# Lecture 4A: Parsing FASTA Files

Converting sequence data into Python dictionaries

## What is FASTA format?

[FASTA](https://en.wikipedia.org/wiki/FASTA_format) is one of the most common formats for storing biological sequences. Each sequence entry consists of:

1. A **header line** starting with `>`, followed by the sequence name and optional description
2. One or more lines containing the **sequence** itself

Here's an example with three protein sequences:

In [ ]:
fasta_data = """>seq1 human insulin
MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT
>seq2 mouse insulin
MALWMRLLPLLALLALWGPEPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPMS
>seq3 zebrafish insulin
MAVWLQAGALLVLLVVSSATSAAPLQTCRKDLQLLKGVVDGLLHTLALALEYKCNTCRGF
"""

print(fasta_data)

> **Note on triple quotes:** In Python, triple quotes (`"""` or `'''`) allow you to create multi-line strings. This is perfect for storing FASTA data directly in your code without needing to read from a file.

## Our Goal: A Dictionary

We want to convert this FASTA data into a Python dictionary where:
- **Keys** are sequence names (e.g., `'seq1'`, `'seq2'`, `'seq3'`)
- **Values** are the corresponding sequences

```python
{
    'seq1': 'MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKT',
    'seq2': 'MALWMRLLPLLALLALWGPEPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPMS',
    'seq3': 'MAVWLQAGALLVLLVVSSATSAAPLQTCRKDLQLLKGVVDGLLHTLALALEYKCNTCRGF'
}
```

Why a dictionary? Because it gives us **fast lookup by name**. If we want the sequence for `seq2`, we simply write `sequences['seq2']`.

## Parsing Logic: Step by Step

### Step 1: Split into lines

First, we split our FASTA string into individual lines:

In [ ]:
lines = fasta_data.strip().split('\n')
for i, line in enumerate(lines):
    print(f"Line {i}: {line}")

### Step 2: Identify header lines

Header lines start with `>`. We can detect them using the `startswith()` method:

In [ ]:
for line in lines:
    if line.startswith('>'):
        print(f"HEADER: {line}")
    else:
        print(f"SEQUENCE: {line}")

### Step 3: Extract the sequence name

The sequence name is the first "word" after the `>`. We can extract it by:
1. Removing the `>` character (using `line[1:]` to skip the first character)
2. Splitting on whitespace and taking the first element

In [ ]:
header = ">seq1 human insulin"

# Remove the '>' character
without_gt = header[1:]
print(f"After removing '>': {without_gt}")

# Split on whitespace
parts = without_gt.split()
print(f"Split into parts: {parts}")

# Take the first part (the name)
name = parts[0]
print(f"Sequence name: {name}")

## First Attempt: A Naive Parser

Now let's put it together. The basic idea is:
- When we see a `>`, save the previous sequence (if any) and start a new one
- When we see a sequence line, append it to the current sequence

Here's our first attempt:

In [ ]:
# CAUTION: This version has a bug!

sequences = {}
current_name = None
current_seq = ""

for line in fasta_data.strip().split('\n'):
    if line.startswith('>'):
        # Save the previous sequence (if there is one)
        if current_name:
            sequences[current_name] = current_seq
        # Start a new sequence
        current_name = line[1:].split()[0]
        current_seq = ""
    else:
        # Append to current sequence
        current_seq += line

print("Sequences found:", list(sequences.keys()))
print(f"Number of sequences: {len(sequences)}")

## The Last Sequence Problem

**Wait! We have 3 sequences in our FASTA data, but only 2 were captured!**

What happened to `seq3`?

Let's trace through the logic:

| Line | Action | `sequences` dict |
|------|--------|------------------|
| `>seq1 human insulin` | Start seq1 | `{}` |
| `MALWMR...PKT` | Append to seq1 | `{}` |
| `>seq2 mouse insulin` | **Save seq1**, start seq2 | `{'seq1': '...'}` |
| `MALWMR...PMS` | Append to seq2 | `{'seq1': '...'}` |
| `>seq3 zebrafish insulin` | **Save seq2**, start seq3 | `{'seq1': '...', 'seq2': '...'}` |
| `MAVWLQ...RGF` | Append to seq3 | `{'seq1': '...', 'seq2': '...'}` |
| **(end of file)** | ??? | **seq3 never saved!** |

The problem: **We only save a sequence when we encounter the NEXT `>`**. But the last sequence has no `>` after it!

## The Fix: Don't Forget the Last Sequence

After the loop ends, we need to check if there's a pending sequence that hasn't been saved yet:

In [ ]:
sequences = {}
current_name = None
current_seq = ""

for line in fasta_data.strip().split('\n'):
    if line.startswith('>'):
        # Save the previous sequence (if there is one)
        if current_name:
            sequences[current_name] = current_seq
        # Start a new sequence
        current_name = line[1:].split()[0]
        current_seq = ""
    else:
        # Append to current sequence
        current_seq += line

# Don't forget the last sequence!
if current_name:
    sequences[current_name] = current_seq

print("Sequences found:", list(sequences.keys()))
print(f"Number of sequences: {len(sequences)}")

Now we have all three sequences! Let's verify:

In [ ]:
for name, seq in sequences.items():
    print(f"{name}: {seq[:20]}... (length: {len(seq)})")

## Wrapping It in a Function

Let's create a reusable function:

In [ ]:
def parse_fasta(fasta_string):
    """
    Parse a FASTA-formatted string into a dictionary.
    
    Args:
        fasta_string: A string containing FASTA-formatted sequences
        
    Returns:
        A dictionary mapping sequence names to sequences
    """
    sequences = {}
    current_name = None
    current_seq = ""
    
    for line in fasta_string.strip().split('\n'):
        if line.startswith('>'):
            if current_name:
                sequences[current_name] = current_seq
            current_name = line[1:].split()[0]
            current_seq = ""
        else:
            current_seq += line
    
    # Don't forget the last sequence!
    if current_name:
        sequences[current_name] = current_seq
    
    return sequences

In [ ]:
# Test our function
result = parse_fasta(fasta_data)
print(result)

## Using the Dictionary

Now we can easily access any sequence by name:

In [ ]:
# Get a specific sequence
print("Human insulin sequence:")
print(result['seq1'])

In [ ]:
# Compare sequence lengths
for name, seq in result.items():
    print(f"{name}: {len(seq)} amino acids")

## Handling Multi-line Sequences

Our parser already handles sequences that span multiple lines. Let's test it:

In [ ]:
multiline_fasta = """>gene1 a long sequence
ATGCATGCATGCATGCATGC
GCTAGCTAGCTAGCTAGCTA
TTTTAAAACCCCGGGG
>gene2 another sequence
AAAAAAAAAA
"""

result = parse_fasta(multiline_fasta)
print(f"gene1 length: {len(result['gene1'])}")
print(f"gene1 sequence: {result['gene1']}")

## Summary

Key takeaways from this lecture:

1. **FASTA format** uses `>` to mark sequence headers
2. **Triple-quoted strings** let us embed multi-line data in Python code
3. **Parsing strategy**: Track the "current" sequence name and content as we iterate through lines
4. **The last sequence problem**: Always check for a pending sequence after the loop ends
5. **Dictionaries** provide a natural way to store name→sequence mappings